In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Work with the ECLE WISE Dataset
See https://wise2.ipac.caltech.edu/docs/release/allwise/expsup/sec2_1a.html for the data description

In [5]:
wise = pd.read_csv('../data/wise-fits-data/all-wise-photometry.csv')

wise.keys()

Index(['designation', 'ra', 'dec', 'sigra', 'sigdec', 'sigradec', 'glon',
       'glat', 'elon', 'elat',
       ...
       'h_m_2mass', 'h_msig_2mass', 'k_m_2mass', 'k_msig_2mass', 'x', 'y', 'z',
       'spt_ind', 'htm20', 'internal_name'],
      dtype='object', length=299)

In [6]:
# 3.4, 4.6, 12 and 22
# from https://www.cambridge.org/core/journals/publications-of-the-astronomical-society-of-australia/article/recalibrating-the-widefield-infrared-survey-explorer-wise-w4-filter/B238BFFE19A533A2D2638FE88CCC2E89
band_vals={1:3.4,2:4.6,3:12,4:22} # in um

dfs = []
for band_val, eff in band_vals.items():
    keep = ['internal_name', f'w{band_val}mjdmean', f'w{band_val}mpro', f'w{band_val}sigmpro', f'w{band_val}snr']
    keep_in_order = ['internal_name', 'date_mjd', 'filter', 'filter_eff', 'filter_eff_unit', f'flux', f'flux_err', 'flux_unit', 'upperlimit']
    
    df = wise[keep]
    
    # this is defined in the WISE data description
    df['upperlimit'] = df.apply(lambda row : row[f'w{band_val}snr'] < 2, axis=1)
    
    # need to then convert upperlimits to 3sigma instead of 2sigma like WISE gives them
    df['flux'] = df.apply(lambda row : 1.5*row[f'w{band_val}mpro'] if row.upperlimit else row[f'w{band_val}mpro'], axis=1)
    df['flux_err'] = df.apply(lambda row : 0 if row.upperlimit else row[f'w{band_val}sigmpro'], axis=1)
    
    df['filter'] = f'W{band_val}'
    df['filter_eff'] = eff
    df['filter_eff_unit'] = 'um'
    df['flux_unit'] = 'mag(AB)'
    
    df['date_mjd'] = df[f'w{band_val}mjdmean']
    
    dfs.append(df[keep_in_order])
    
wise_cleaned = pd.concat(dfs)

wise_cleaned.to_csv('cleaned-wise-ecle-data.csv')

/tmp/ipykernel_773539/1513521362.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['upperlimit'] = df.apply(lambda row : row[f'w{band_val}snr'] < 2, axis=1)
/tmp/ipykernel_773539/1513521362.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['flux'] = df.apply(lambda row : 1.5*row[f'w{band_val}mpro'] if row.upperlimit else row[f'w{band_val}mpro'], axis=1)
/tmp/ipykernel_773539/1513521362.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

In [7]:
wise_cleaned

,internal_name,date_mjd,filter,filter_eff,filter_eff_unit,flux,flux_err,flux_unit,upperlimit
0,SDSS_J0748,55390.796856,W1,3.4,um,13.5050,0.025,mag(AB),False
1,SDSS_J0807,55401.001991,W1,3.4,um,11.6460,0.023,mag(AB),False
2,SDSS_J0938,55419.361618,W1,3.4,um,11.9820,0.023,mag(AB),False
3,SDSS_J0952,55405.204708,W1,3.4,um,13.6660,0.026,mag(AB),False
4,SDSS_J1055,55405.695265,W1,3.4,um,13.6810,0.024,mag(AB),False
5,SDSS_J1207,55441.162975,W1,3.4,um,11.8360,0.023,mag(AB),False
6,SDSS_J1241,55427.227305,W1,3.4,um,13.8540,0.028,mag(AB),False
7,SDSS_J1247,55463.103257,W1,3.4,um,13.1670,0.023,mag(AB),False
8,SDSS_J1342,55419.290861,W1,3.4,um,13.4280,0.025,mag(AB),False
9,SDSS_J1350,55416.193977,W1,3.4,um,13.9250,0.025,mag(AB),False
